# Data Preparation
## Part 1: Prepare SATCAT Data
#### 1. Acquire all debris data from SATCAT via API from [space-track.org](https://www.space-track.org/) using the [spacetrack module](https://spacetrack.readthedocs.io/en/stable/)

In [1]:
# Space-Track.org API authentication 
from spacetrack import SpaceTrackClient

# read Space-Track.org credentials from external file (username in line 1, password in line 2)
with open('../DS_SpaceTrackCredentials.txt', 'r') as file:
    credentials = file.read().splitlines()

username = credentials[0]
password = credentials[1]

# create a SpaceTrackClient instance
st = SpaceTrackClient(username, password) 

try:
    # Try fetching a single SATCAT entry to verify authentication
    test_data = st.satcat(limit=1, format='json')
    print("Successfully authenticated with Space-Track.org!")
except Exception as e:
    print(f"Failed to authenticate with Space-Track.org: {e}")

print("Test data: " + test_data)

Successfully authenticated with Space-Track.org!
Test data: [{"INTLDES":"1957-001A","NORAD_CAT_ID":"1","OBJECT_TYPE":"ROCKET BODY","SATNAME":"SL-1 R\/B","COUNTRY":"CIS","LAUNCH":"1957-10-04","SITE":"TTMTR","DECAY":"1957-12-01","PERIOD":"96.19","INCLINATION":"65.10","APOGEE":"938","PERIGEE":"214","COMMENT":null,"COMMENTCODE":"4","RCSVALUE":"0","RCS_SIZE":"LARGE","FILE":"1","LAUNCH_YEAR":"1957","LAUNCH_NUM":"1","LAUNCH_PIECE":"A","CURRENT":"Y","OBJECT_NAME":"SL-1 R\/B","OBJECT_ID":"1957-001A","OBJECT_NUMBER":"1"}]


In [2]:
# Fetch all SATCAT data from Space-Track
import json
import pandas as pd

satcat_data = st.satcat(format='json') # Fetch all SATCAT data from Space-Track in JSON format
print("Datatype: " + str(type(satcat_data))) # Check the datatype of the SATCAT data
satcat_data = json.loads(satcat_data) # Convert JSON string to a Python list of dictionaries
satcat_df = pd.DataFrame(satcat_data) # Convert list of dictionaries to a Pandas DataFrame
satcat_df

Datatype: <class 'str'>


,INTLDES,NORAD_CAT_ID,OBJECT_TYPE,SATNAME,COUNTRY,LAUNCH,SITE,DECAY,PERIOD,INCLINATION,...,RCSVALUE,RCS_SIZE,FILE,LAUNCH_YEAR,LAUNCH_NUM,LAUNCH_PIECE,CURRENT,OBJECT_NAME,OBJECT_ID,OBJECT_NUMBER
0,1957-001A,1,ROCKET BODY,SL-1 R/B,CIS,1957-10-04,TTMTR,1957-12-01,96.19,65.10,...,0,LARGE,1,1957,1,A,Y,SL-1 R/B,1957-001A,1
1,1958-001A,4,PAYLOAD,EXPLORER 1,US,1958-02-01,AFETR,1970-03-31,88.48,33.15,...,0,None,1,1958,1,A,Y,EXPLORER 1,1958-001A,4
2,1958-003A,6,PAYLOAD,EXPLORER 3,US,1958-03-26,AFETR,1958-06-28,103.60,33.50,...,0,None,1,1958,3,A,Y,EXPLORER 3,1958-003A,6
3,1958-004A,7,ROCKET BODY,SL-1 R/B,CIS,1958-05-15,TTMTR,1958-12-03,102.74,65.14,...,0,None,1,1958,4,A,Y,SL-1 R/B,1958-004A,7
4,1958-004B,8,PAYLOAD,SPUTNIK 3,CIS,1958-05-15,TTMTR,1960-04-06,88.43,65.06,...,0,LARGE,1,1958,4,B,Y,SPUTNIK 3,1958-004B,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62642,2025-018W,62826,PAYLOAD,STARLINK-32704,US,2025-01-24,AFWTR,None,90.69,53.15,...,0,None,9031,2025,18,W,Y,STARLINK-32704,2025-018W,62826
62643,2025-018X,62827,PAYLOAD,STARLINK-32838,US,2025-01-24,AFWTR,None,90.34,53.15,...,0,None,9031,2025,18,X,Y,STARLINK-32838,2025-018X,62827
62644,2025-018Y,62828,PAYLOAD,STARLINK-32727,US,2025-01-24,AFWTR,None,90.68,53.15,...,0,None,9031,2025,18,Y,Y,STARLINK-32727,2025-018Y,62828
62645,2025-020A,62850,UNKNOWN,OBJECT A,IND,2025-01-29,SRI,None,665.11,20.77,...,0,LARGE,9031,2025,20,A,Y,OBJECT A,2025-020A,62850


#### 2. Filter Data for only non operating space objects (debris and rocket bodies) that have decayed and have a low earth orbit 

In [3]:
# Check unique values in OBJECT_TYPE column
object_type_counts = satcat_df["OBJECT_TYPE"].value_counts()
print(object_type_counts)

OBJECT_TYPE
DEBRIS         35568
PAYLOAD        19570
ROCKET BODY     6658
UNKNOWN          851
Name: count, dtype: int64


In [4]:
# extract only debris and rocket body objects
debris_df = satcat_df[satcat_df["OBJECT_TYPE"].isin(["DEBRIS", "ROCKET BODY"])]
debris_df

,INTLDES,NORAD_CAT_ID,OBJECT_TYPE,SATNAME,COUNTRY,LAUNCH,SITE,DECAY,PERIOD,INCLINATION,...,RCSVALUE,RCS_SIZE,FILE,LAUNCH_YEAR,LAUNCH_NUM,LAUNCH_PIECE,CURRENT,OBJECT_NAME,OBJECT_ID,OBJECT_NUMBER
0,1957-001A,1,ROCKET BODY,SL-1 R/B,CIS,1957-10-04,TTMTR,1957-12-01,96.19,65.10,...,0,LARGE,1,1957,1,A,Y,SL-1 R/B,1957-001A,1
3,1958-004A,7,ROCKET BODY,SL-1 R/B,CIS,1958-05-15,TTMTR,1958-12-03,102.74,65.14,...,0,None,1,1958,4,A,Y,SL-1 R/B,1958-004A,7
10,1959-004B,17,ROCKET BODY,THOR ABLE R/B,US,1959-08-07,AFETR,1961-06-30,718.96,47.10,...,0,SMALL,1,1959,4,B,Y,THOR ABLE R/B,1959-004B,17
17,1960-003A,30,ROCKET BODY,THOR ABLESTAR R/B,US,1960-04-13,AFETR,1961-08-18,88.38,51.25,...,0,None,1,1960,3,A,Y,THOR ABLESTAR R/B,1960-003A,30
20,1960-003C,33,DEBRIS,THOR ABLESTAR DEB,US,1960-04-13,AFETR,1960-07-17,93.59,51.29,...,0,None,1,1960,3,C,Y,THOR ABLESTAR DEB,1960-003C,33
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62490,2024-250B,62431,ROCKET BODY,SL-4 R/B,CIS,2024-12-25,TTMTR,None,91.20,97.27,...,0,LARGE,9031,2024,250,B,Y,SL-4 R/B,2024-250B,62431
62505,2024-252E,62458,ROCKET BODY,FALCON 9 R/B,US,2024-12-29,AFETR,None,1238.38,12.40,...,0,None,9031,2024,252,E,Y,FALCON 9 R/B,2024-252E,62458
62516,2025-002B,62486,ROCKET BODY,CZ-3B R/B,PRC,2025-01-06,XSC,None,642.54,28.57,...,0,LARGE,9031,2025,2,B,Y,CZ-3B R/B,2025-002B,62486
62549,2025-007M,62586,ROCKET BODY,JIELONG 3 R/B,PRC,2025-01-13,YSLA,None,91.97,54.94,...,0,LARGE,9031,2025,7,M,Y,JIELONG 3 R/B,2025-007M,62586


In [5]:
# Show values in DECAY column
print(debris_df['DECAY'].unique().tolist())

['1957-12-01', '1958-12-03', '1961-06-30', '1961-08-18', '1960-07-17', '1960-08-20', '1960-09-24', '1960-09-30', '1960-10-01', '1960-12-05', '1960-09-23', '1960-12-02', '1970-02-16', '1961-02-13', '1961-03-17', '1961-02-18', '1970-10-09', '1961-04-02', '1961-04-20', '1961-10-31', '1961-03-10', '1961-03-26', '1961-04-16', '1972-03-03', '1961-09-10', '1961-06-19', '1962-09-30', '1967-12-25', '1962-06-16', '1969-07-07', '1967-12-02', '1981-01-08', '1961-07-24', '1961-08-09', '1962-01-29', '1961-09-03', '1961-09-13', '1961-09-18', '1961-09-28', '1961-11-25', '1961-10-16', '1961-12-05', '1961-11-30', '1961-12-09', '1961-12-10', '1961-12-12', '1961-11-23', '1988-08-11', '1989-11-13', '1961-12-31', '1962-01-09', '1966-02-21', '1965-02-13', '1969-05-13', '1969-04-12', '1969-12-07', '1967-06-17', '1962-02-21', '1962-03-03', '1962-03-07', '1970-05-03', '1962-03-31', '1962-11-03', '1962-06-18', '1962-10-06', '1962-05-04', '1962-04-20', '1962-04-21', '1962-08-05', '1962-06-17', '1971-04-28', '1962

In [6]:
# Filter only decayed objects
decayedDebris_df = debris_df[debris_df["DECAY"].notnull()]
decayedDebris_df

,INTLDES,NORAD_CAT_ID,OBJECT_TYPE,SATNAME,COUNTRY,LAUNCH,SITE,DECAY,PERIOD,INCLINATION,...,RCSVALUE,RCS_SIZE,FILE,LAUNCH_YEAR,LAUNCH_NUM,LAUNCH_PIECE,CURRENT,OBJECT_NAME,OBJECT_ID,OBJECT_NUMBER
0,1957-001A,1,ROCKET BODY,SL-1 R/B,CIS,1957-10-04,TTMTR,1957-12-01,96.19,65.10,...,0,LARGE,1,1957,1,A,Y,SL-1 R/B,1957-001A,1
3,1958-004A,7,ROCKET BODY,SL-1 R/B,CIS,1958-05-15,TTMTR,1958-12-03,102.74,65.14,...,0,None,1,1958,4,A,Y,SL-1 R/B,1958-004A,7
10,1959-004B,17,ROCKET BODY,THOR ABLE R/B,US,1959-08-07,AFETR,1961-06-30,718.96,47.10,...,0,SMALL,1,1959,4,B,Y,THOR ABLE R/B,1959-004B,17
17,1960-003A,30,ROCKET BODY,THOR ABLESTAR R/B,US,1960-04-13,AFETR,1961-08-18,88.38,51.25,...,0,None,1,1960,3,A,Y,THOR ABLESTAR R/B,1960-003A,30
20,1960-003C,33,DEBRIS,THOR ABLESTAR DEB,US,1960-04-13,AFETR,1960-07-17,93.59,51.29,...,0,None,1,1960,3,C,Y,THOR ABLESTAR DEB,1960-003C,33
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50117,2024-140WZ,61384,DEBRIS,CZ-6A DEB,PRC,2024-08-06,TSC,2024-11-13,88.12,88.95,...,0,SMALL,9029,2024,140,WZ,Y,CZ-6A DEB,2024-140WZ,61384
50124,2024-140XY,61407,DEBRIS,CZ-6A DEB,PRC,2024-08-06,TSC,2024-12-17,90.54,89.21,...,0,SMALL,9029,2024,140,XY,Y,CZ-6A DEB,2024-140XY,61407
50132,2024-140ZF,61438,DEBRIS,CZ-6A DEB,PRC,2024-08-06,TSC,2024-11-16,88.52,90.08,...,0,SMALL,9029,2024,140,ZF,Y,CZ-6A DEB,2024-140ZF,61438
50134,2024-177B,61445,ROCKET BODY,CZ-2D R/B,PRC,2024-09-27,JSC,2024-11-10,87.91,41.38,...,0,LARGE,9029,2024,177,B,Y,CZ-2D R/B,2024-177B,61445


In [7]:
# test if apogee is always bigger than perigee
perigee_higher = decayedDebris_df[decayedDebris_df["APOGEE"] < decayedDebris_df["PERIGEE"]]
print(perigee_higher[['APOGEE', 'PERIGEE']])
print(decayedDebris_df[['APOGEE', 'PERIGEE']].dtypes)


      APOGEE PERIGEE
3       1571     206
81       190      93
144     2729      99
278      421      77
597      214      87
...      ...     ...
40590   2076     401
40596   3227     854
45152   2026      97
47439   1036     496
47494   1104     808

[1056 rows x 2 columns]
APOGEE     object
PERIGEE    object
dtype: object


In [8]:
# Convert APOGEE and PERIGEE columns from type object to numeric types since some where not compared correctly
decayedDebris_df['APOGEE'] = pd.to_numeric(decayedDebris_df['APOGEE'], errors='coerce')
decayedDebris_df['PERIGEE'] = pd.to_numeric(decayedDebris_df['PERIGEE'], errors='coerce')


/var/folders/xm/n65_n9sd78z4_zdw8r87klf00000gn/T/ipykernel_3103/1945170451.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  decayedDebris_df['APOGEE'] = pd.to_numeric(decayedDebris_df['APOGEE'], errors='coerce')
/var/folders/xm/n65_n9sd78z4_zdw8r87klf00000gn/T/ipykernel_3103/1945170451.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  decayedDebris_df['PERIGEE'] = pd.to_numeric(decayedDebris_df['PERIGEE'], errors='coerce')


In [9]:
# test again if apogee is always bigger than perigee
perigee_higher_repeat = (decayedDebris_df["APOGEE"] < decayedDebris_df["PERIGEE"]).all()
print(perigee_higher_repeat)

False


In [10]:
# Filter only objects that are in Low Earth Orbit
# LEO is defined as an orbit with an apogee lower than 2000 km
decayedLEODebris_df = decayedDebris_df[decayedDebris_df["APOGEE"] < 2000].copy()
decayedLEODebris_df

,INTLDES,NORAD_CAT_ID,OBJECT_TYPE,SATNAME,COUNTRY,LAUNCH,SITE,DECAY,PERIOD,INCLINATION,...,RCSVALUE,RCS_SIZE,FILE,LAUNCH_YEAR,LAUNCH_NUM,LAUNCH_PIECE,CURRENT,OBJECT_NAME,OBJECT_ID,OBJECT_NUMBER
0,1957-001A,1,ROCKET BODY,SL-1 R/B,CIS,1957-10-04,TTMTR,1957-12-01,96.19,65.10,...,0,LARGE,1,1957,1,A,Y,SL-1 R/B,1957-001A,1
3,1958-004A,7,ROCKET BODY,SL-1 R/B,CIS,1958-05-15,TTMTR,1958-12-03,102.74,65.14,...,0,None,1,1958,4,A,Y,SL-1 R/B,1958-004A,7
17,1960-003A,30,ROCKET BODY,THOR ABLESTAR R/B,US,1960-04-13,AFETR,1961-08-18,88.38,51.25,...,0,None,1,1960,3,A,Y,THOR ABLESTAR R/B,1960-003A,30
20,1960-003C,33,DEBRIS,THOR ABLESTAR DEB,US,1960-04-13,AFETR,1960-07-17,93.59,51.29,...,0,None,1,1960,3,C,Y,THOR ABLESTAR DEB,1960-003C,33
22,1960-005B,35,ROCKET BODY,SL-3 R/B,CIS,1960-05-15,TTMTR,1960-07-17,88.48,64.89,...,0,None,1,1960,5,B,Y,SL-3 R/B,1960-005B,35
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50117,2024-140WZ,61384,DEBRIS,CZ-6A DEB,PRC,2024-08-06,TSC,2024-11-13,88.12,88.95,...,0,SMALL,9029,2024,140,WZ,Y,CZ-6A DEB,2024-140WZ,61384
50124,2024-140XY,61407,DEBRIS,CZ-6A DEB,PRC,2024-08-06,TSC,2024-12-17,90.54,89.21,...,0,SMALL,9029,2024,140,XY,Y,CZ-6A DEB,2024-140XY,61407
50132,2024-140ZF,61438,DEBRIS,CZ-6A DEB,PRC,2024-08-06,TSC,2024-11-16,88.52,90.08,...,0,SMALL,9029,2024,140,ZF,Y,CZ-6A DEB,2024-140ZF,61438
50134,2024-177B,61445,ROCKET BODY,CZ-2D R/B,PRC,2024-09-27,JSC,2024-11-10,87.91,41.38,...,0,LARGE,9029,2024,177,B,Y,CZ-2D R/B,2024-177B,61445


#### 3. Delete not needed columns

In [11]:
# show all columns
pd.set_option('display.max_columns', None)
decayedLEODebris_df

,INTLDES,NORAD_CAT_ID,OBJECT_TYPE,SATNAME,COUNTRY,LAUNCH,SITE,DECAY,PERIOD,INCLINATION,APOGEE,PERIGEE,COMMENT,COMMENTCODE,RCSVALUE,RCS_SIZE,FILE,LAUNCH_YEAR,LAUNCH_NUM,LAUNCH_PIECE,CURRENT,OBJECT_NAME,OBJECT_ID,OBJECT_NUMBER
0,1957-001A,1,ROCKET BODY,SL-1 R/B,CIS,1957-10-04,TTMTR,1957-12-01,96.19,65.10,938.0,214.0,None,4,0,LARGE,1,1957,1,A,Y,SL-1 R/B,1957-001A,1
3,1958-004A,7,ROCKET BODY,SL-1 R/B,CIS,1958-05-15,TTMTR,1958-12-03,102.74,65.14,1571.0,206.0,None,None,0,None,1,1958,4,A,Y,SL-1 R/B,1958-004A,7
17,1960-003A,30,ROCKET BODY,THOR ABLESTAR R/B,US,1960-04-13,AFETR,1961-08-18,88.38,51.25,194.0,194.0,None,None,0,None,1,1960,3,A,Y,THOR ABLESTAR R/B,1960-003A,30
20,1960-003C,33,DEBRIS,THOR ABLESTAR DEB,US,1960-04-13,AFETR,1960-07-17,93.59,51.29,615.0,285.0,None,None,0,None,1,1960,3,C,Y,THOR ABLESTAR DEB,1960-003C,33
22,1960-005B,35,ROCKET BODY,SL-3 R/B,CIS,1960-05-15,TTMTR,1960-07-17,88.48,64.89,199.0,199.0,None,None,0,None,1,1960,5,B,Y,SL-3 R/B,1960-005B,35
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50117,2024-140WZ,61384,DEBRIS,CZ-6A DEB,PRC,2024-08-06,TSC,2024-11-13,88.12,88.95,191.0,172.0,None,None,0,SMALL,9029,2024,140,WZ,Y,CZ-6A DEB,2024-140WZ,61384
50124,2024-140XY,61407,DEBRIS,CZ-6A DEB,PRC,2024-08-06,TSC,2024-12-17,90.54,89.21,313.0,288.0,None,None,0,SMALL,9029,2024,140,XY,Y,CZ-6A DEB,2024-140XY,61407
50132,2024-140ZF,61438,DEBRIS,CZ-6A DEB,PRC,2024-08-06,TSC,2024-11-16,88.52,90.08,221.0,181.0,None,None,0,SMALL,9029,2024,140,ZF,Y,CZ-6A DEB,2024-140ZF,61438
50134,2024-177B,61445,ROCKET BODY,CZ-2D R/B,PRC,2024-09-27,JSC,2024-11-10,87.91,41.38,176.0,167.0,None,None,0,LARGE,9029,2024,177,B,Y,CZ-2D R/B,2024-177B,61445


In [12]:
# perform tests to see which columns can be deleted
print(decayedLEODebris_df['CURRENT'].unique())
print(decayedLEODebris_df['INTLDES'].equals(decayedLEODebris_df['OBJECT_ID']))
print(decayedLEODebris_df['SATNAME'].equals(decayedLEODebris_df['OBJECT_NAME']))


['Y']
True
True


In [14]:
decayedLEODebris_df.drop(columns=['COMMENT', 'COMMENTCODE', 'RCSVALUE', 'FILE', 'LAUNCH_YEAR', 'LAUNCH_NUM', 'LAUNCH_PIECE', 'CURRENT', 'OBJECT_ID', 'OBJECT_NAME', 'OBJECT_NUMBER'], inplace=True)
decayedLEODebris_df

,INTLDES,NORAD_CAT_ID,OBJECT_TYPE,SATNAME,COUNTRY,LAUNCH,SITE,DECAY,PERIOD,INCLINATION,APOGEE,PERIGEE,RCS_SIZE
0,1957-001A,1,ROCKET BODY,SL-1 R/B,CIS,1957-10-04,TTMTR,1957-12-01,96.19,65.10,938.0,214.0,LARGE
3,1958-004A,7,ROCKET BODY,SL-1 R/B,CIS,1958-05-15,TTMTR,1958-12-03,102.74,65.14,1571.0,206.0,None
17,1960-003A,30,ROCKET BODY,THOR ABLESTAR R/B,US,1960-04-13,AFETR,1961-08-18,88.38,51.25,194.0,194.0,None
20,1960-003C,33,DEBRIS,THOR ABLESTAR DEB,US,1960-04-13,AFETR,1960-07-17,93.59,51.29,615.0,285.0,None
22,1960-005B,35,ROCKET BODY,SL-3 R/B,CIS,1960-05-15,TTMTR,1960-07-17,88.48,64.89,199.0,199.0,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...
50117,2024-140WZ,61384,DEBRIS,CZ-6A DEB,PRC,2024-08-06,TSC,2024-11-13,88.12,88.95,191.0,172.0,SMALL
50124,2024-140XY,61407,DEBRIS,CZ-6A DEB,PRC,2024-08-06,TSC,2024-12-17,90.54,89.21,313.0,288.0,SMALL
50132,2024-140ZF,61438,DEBRIS,CZ-6A DEB,PRC,2024-08-06,TSC,2024-11-16,88.52,90.08,221.0,181.0,SMALL
50134,2024-177B,61445,ROCKET BODY,CZ-2D R/B,PRC,2024-09-27,JSC,2024-11-10,87.91,41.38,176.0,167.0,LARGE


In [15]:
# save to csv
decayedLEODebris_df.to_csv('../Data/DS_DecayedLEODebris.csv', index=False)
decayedLEODebris_df['NORAD_CAT_ID'].to_csv('../Data/DS_NoradIDs.csv', index=False)

#### 4. Inspect filtered data

In [15]:
import pandas as pd

# Load and check data to avoid having to download and peform filtering all again
decayedLEODebris_df = pd.read_csv("../Data/DS_decayedLEODebris.csv")
decayedLEODebris_df

,INTLDES,NORAD_CAT_ID,OBJECT_TYPE,SATNAME,COUNTRY,LAUNCH,SITE,DECAY,PERIOD,INCLINATION,APOGEE,PERIGEE,RCS_SIZE,LAUNCH_YEAR,LAUNCH_NUM,LAUNCH_PIECE,OBJECT_NUMBER
0,1957-001A,1,ROCKET BODY,SL-1 R/B,CIS,1957-10-04,TTMTR,1957-12-01,96.19,65.10,938.0,214.0,LARGE,1957,1,A,1
1,1958-004A,7,ROCKET BODY,SL-1 R/B,CIS,1958-05-15,TTMTR,1958-12-03,102.74,65.14,1571.0,206.0,NaN,1958,4,A,7
2,1960-003A,30,ROCKET BODY,THOR ABLESTAR R/B,US,1960-04-13,AFETR,1961-08-18,88.38,51.25,194.0,194.0,NaN,1960,3,A,30
3,1960-003C,33,DEBRIS,THOR ABLESTAR DEB,US,1960-04-13,AFETR,1960-07-17,93.59,51.29,615.0,285.0,NaN,1960,3,C,33
4,1960-005B,35,ROCKET BODY,SL-3 R/B,CIS,1960-05-15,TTMTR,1960-07-17,88.48,64.89,199.0,199.0,NaN,1960,5,B,35
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25695,2024-140WZ,61384,DEBRIS,CZ-6A DEB,PRC,2024-08-06,TSC,2024-11-13,88.12,88.95,191.0,172.0,SMALL,2024,140,WZ,61384
25696,2024-140XY,61407,DEBRIS,CZ-6A DEB,PRC,2024-08-06,TSC,2024-12-17,90.54,89.21,313.0,288.0,SMALL,2024,140,XY,61407
25697,2024-140ZF,61438,DEBRIS,CZ-6A DEB,PRC,2024-08-06,TSC,2024-11-16,88.52,90.08,221.0,181.0,SMALL,2024,140,ZF,61438
25698,2024-177B,61445,ROCKET BODY,CZ-2D R/B,PRC,2024-09-27,JSC,2024-11-10,87.91,41.38,176.0,167.0,LARGE,2024,177,B,61445


In [16]:
# get basic statistics of numerical columns
print(decayedLEODebris_df.describe())

       NORAD_CAT_ID        PERIOD   INCLINATION        APOGEE       PERIGEE  \
count  25700.000000  25700.000000  25700.000000  25700.000000  25700.000000   
mean   23517.787549     90.498625     71.489491    351.459105    254.748405   
std    15429.016689      4.614714     20.196929    240.525378    130.186964   
min        1.000000      0.000000      0.000000      0.000000      0.000000   
25%    10672.500000     88.300000     62.750000    206.000000    162.000000   
50%    20943.500000     89.580000     72.780000    283.000000    211.000000   
75%    34424.500000     91.790000     82.890000    404.000000    315.000000   
max    62608.000000    116.890000    150.940000   1998.000000   1478.000000   

        LAUNCH_YEAR    LAUNCH_NUM  OBJECT_NUMBER  
count  25700.000000  25700.000000   25700.000000  
mean    1987.255525     53.877276   23517.787549  
std       14.742089     33.065076   15429.016689  
min     1957.000000      0.000000       1.000000  
25%     1976.000000     25.000000

In [17]:
# Check for missing values
print(decayedLEODebris_df.isnull().sum())

INTLDES             0
NORAD_CAT_ID        0
OBJECT_TYPE         0
SATNAME             0
COUNTRY             0
LAUNCH              0
SITE                0
DECAY               0
PERIOD              0
INCLINATION         0
APOGEE              0
PERIGEE             0
RCS_SIZE         6894
LAUNCH_YEAR         0
LAUNCH_NUM          0
LAUNCH_PIECE       10
OBJECT_NUMBER       0
dtype: int64


In [18]:
# Data distribution
print(decayedLEODebris_df["OBJECT_TYPE"].value_counts())  # Number of debris vs. rocket bodies
print(decayedLEODebris_df["COUNTRY"].value_counts().head(10))  # Top 10 countries with most debris
print(decayedLEODebris_df["RCS_SIZE"].value_counts(dropna=False))  # Radar Cross Section size

OBJECT_TYPE
DEBRIS         21709
ROCKET BODY     3991
Name: count, dtype: int64
COUNTRY
CIS     15497
US       6047
PRC      2219
FR        740
IND       515
JPN       413
ISS       100
CHBZ       75
ESA        31
ORB        27
Name: count, dtype: int64
RCS_SIZE
SMALL     12051
NaN        6894
MEDIUM     3705
LARGE      3050
Name: count, dtype: int64


In [19]:
# show columns
print(decayedLEODebris_df.columns)

Index(['INTLDES', 'NORAD_CAT_ID', 'OBJECT_TYPE', 'SATNAME', 'COUNTRY',
       'LAUNCH', 'SITE', 'DECAY', 'PERIOD', 'INCLINATION', 'APOGEE', 'PERIGEE',
       'RCS_SIZE', 'LAUNCH_YEAR', 'LAUNCH_NUM', 'LAUNCH_PIECE',
       'OBJECT_NUMBER'],
      dtype='object')
